# Auditoria: Frete de Coleta vs Frete Proprio

Confere se cada lancamento de **Frete Proprio** (`5.7.2`, receita) tem o **Frete de Coleta** (`6.6.6`, despesa) correspondente, e vice-versa.

## Chave de relacionamento

O numero do CTE da coleta esta embutido no CTF do frete proprio com **um digito de prefixo regional**:

| Filial Coleta             | Exemplo CTE  | Exemplo CTF   | Prefixo |
|---------------------------|--------------|---------------|---------|
| G3S PRUDENTE              | `CTE-13481`  | `CTF013481`   | `0`     |
| G3S MARINGA / LONDRINA    | `CTE-5287`   | `CTF55287`    | `5`     |
| G3S DOURADOS / CAMPO GRANDE | `CTE-2500` | `CTF42500`    | `4`     |

**Regra:**
- Coleta: tira `CTE-` -> numero
- Proprio: tira `CTF` + remove o 1o digito -> numero

In [38]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

BASE = Path('../../02-Referencias/Meus_Dados')
COLETA_PATH = BASE / 'Frete_Coleta.csv'
PROPRIO_PATH = BASE / 'Frete_Proprio.csv'

print('Coleta :', COLETA_PATH.resolve())
print('Proprio:', PROPRIO_PATH.resolve())

Coleta : C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Meus_Dados\Frete_Coleta.csv
Proprio: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Meus_Dados\Frete_Proprio.csv


## 1. Carga e inspecao

In [39]:
def ler_csv(path):
    for enc in ('utf-8', 'utf-8-sig', 'latin-1', 'cp1252'):
        try:
            return pd.read_csv(path, sep=';', decimal=',', dtype=str, encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f'Nao foi possivel decodificar {path}')

df_coleta, enc_c = ler_csv(COLETA_PATH)
df_proprio, enc_p = ler_csv(PROPRIO_PATH)
print(f'Encoding Coleta : {enc_c}')
print(f'Encoding Proprio: {enc_p}')

for col in ['valor_plano', 'valor_centro', 'iterea_valpago', 'valor_bruto']:
    df_coleta[col] = pd.to_numeric(df_coleta[col].str.replace(',', '.'), errors='coerce')
    df_proprio[col] = pd.to_numeric(df_proprio[col].str.replace(',', '.'), errors='coerce')

print('\nFrete Coleta :', df_coleta.shape)
print('Frete Proprio:', df_proprio.shape)
print('\nColunas Coleta :', list(df_coleta.columns))
print('Colunas Proprio:', list(df_proprio.columns))

Encoding Coleta : latin-1
Encoding Proprio: utf-8

Frete Coleta : (153, 18)
Frete Proprio: (163, 18)

Colunas Coleta : ['codcen', 'descen', 'codcdc', 'descdc', 'lancamento', 'ite_pagrec_vencimento', 'iterea_pagamento', 'iterea_valpago', 'documento', 'codigo_pessoa', 'nome', 'valor_plano', 'valor_centro', 'observacao', 'nota', 'cab_pagrec_id', 'valor_bruto', 'filial']
Colunas Proprio: ['codcen', 'descen', 'codcdc', 'descdc', 'lancamento', 'ite_pagrec_vencimento', 'iterea_pagamento', 'iterea_valpago', 'documento', 'codigo_pessoa', 'nome', 'valor_plano', 'valor_centro', 'observacao', 'nota', 'cab_pagrec_id', 'valor_bruto', 'filial']


In [40]:
df_coleta.head(3)

,codcen,descen,codcdc,descdc,lancamento,ite_pagrec_vencimento,iterea_pagamento,iterea_valpago,documento,codigo_pessoa,nome,valor_plano,valor_centro,observacao,nota,cab_pagrec_id,valor_bruto,filial
0,1.2.4.2,GERAL CONSOLIDADO / SELETIVA / MARINGA / COMER...,6.6.6,FRETE DE COLETA,30/03/2026,2026-03-30,NAO PAGO,0,CTE-5287,7886,G&S TRANSPORTE LOGISTICA E LOCACAO DE MAQUINAS...,-2312.0,-2312.0,- DOCUMENTO ELETRÔNICO REFERENCIADO,5287-2/57,56392,2312.0,G3S MARINGA
1,1.2.5.2,GERAL CONSOLIDADO / SELETIVA / PRESIDENTE PRUD...,6.6.6,FRETE DE COLETA,23/03/2026,2026-04-05,2026-04-24,1890,CTE-13481,376,G&S TRANSPORTE LOGISTICA E LOCACAO DE MAQUINAS...,-1890.0,-1890.0,- DOCUMENTO ELETRÔNICO REFERENCIADO,13481-1/57,55883,1890.0,G3S PRUDENTE
2,1.2.5.2,GERAL CONSOLIDADO / SELETIVA / PRESIDENTE PRUD...,6.6.6,FRETE DE COLETA,23/03/2026,2026-04-05,2026-04-24,2254,CTE-13475,376,G&S TRANSPORTE LOGISTICA E LOCACAO DE MAQUINAS...,-2254.0,-2254.0,- DOCUMENTO ELETRÔNICO REFERENCIADO,13475-1/57,55945,2254.0,G3S PRUDENTE


In [41]:
df_proprio.head(3)

,codcen,descen,codcdc,descdc,lancamento,ite_pagrec_vencimento,iterea_pagamento,iterea_valpago,documento,codigo_pessoa,nome,valor_plano,valor_centro,observacao,nota,cab_pagrec_id,valor_bruto,filial
0,2.7.8.2,RECEITA / EKIPA SERVICOS - G&S / G&S PRUDENTE ...,5.7.2,FRETE PROPRIO,23/03/2026,2026-03-23,2026-04-24,2254.0,CTF013475,330,G3S COMERCIO E INDUSTRIA DE FERRO E ACO LTDA,2254.0,2254.0,,NaN,54951,2254.0,G&S PRUDENTE
1,2.7.8.2,RECEITA / EKIPA SERVICOS - G&S / G&S PRUDENTE ...,5.7.2,FRETE PROPRIO,23/03/2026,2026-03-23,2026-04-24,2338.0,CTF013480,330,G3S COMERCIO E INDUSTRIA DE FERRO E ACO LTDA,2338.0,2338.0,,8191-1/FL,54952,2338.0,G&S PRUDENTE
2,2.7.8.2,RECEITA / EKIPA SERVICOS - G&S / G&S PRUDENTE ...,5.7.2,FRETE PROPRIO,23/03/2026,2026-03-23,2026-04-24,1890.0,CTF013481,330,G3S COMERCIO E INDUSTRIA DE FERRO E ACO LTDA,1890.0,1890.0,,8192-1/FL,54954,1890.0,G&S PRUDENTE


In [42]:
print('Registros por filial - COLETA')
print(df_coleta['filial'].value_counts())
print('\nRegistros por filial - PROPRIO')
print(df_proprio['filial'].value_counts())

Registros por filial - COLETA
filial
G3S MARINGA         56
G3S PRUDENTE        41
G3S CAMPO GRANDE    28
G3S DOURADOS        14
G3S LONDRINA        14
Name: count, dtype: int64

Registros por filial - PROPRIO
filial
G&S MARINGA     69
G&S DOURADOS    55
G&S PRUDENTE    39
Name: count, dtype: int64


## 2. Extracao da chave de documento

- Coleta: `CTE-NNNN` -> `NNNN`
- Proprio: `CTFXNNNN` -> `NNNN` (descarta `CTF` + 1 digito de prefixo regional)

In [43]:
def key_from_cte(s: str):
    if not isinstance(s, str):
        return None
    s = s.strip()
    if s.startswith('CTE-'):
        return s.replace('CTE-', '').lstrip('0') or '0'
    return None

def key_from_ctf(s: str):
    if not isinstance(s, str):
        return None
    s = s.strip()
    if s.startswith('CTF'):
        rest = s[3:]
        if len(rest) <= 1:
            return None
        return rest[1:].lstrip('0') or '0'
    return None

df_coleta['key'] = df_coleta['documento'].apply(key_from_cte)
df_proprio['key'] = df_proprio['documento'].apply(key_from_ctf)

print('Chaves Coleta  - nulas:', df_coleta['key'].isna().sum())
print('Chaves Proprio - nulas:', df_proprio['key'].isna().sum())

df_coleta[['documento', 'key', 'filial']].head()

Chaves Coleta  - nulas: 0
Chaves Proprio - nulas: 0


,documento,key,filial
0,CTE-5287,5287,G3S MARINGA
1,CTE-13481,13481,G3S PRUDENTE
2,CTE-13475,13475,G3S PRUDENTE
3,CTE-13480,13480,G3S PRUDENTE
4,CTE-13494,13494,G3S PRUDENTE


In [44]:
dup_col = df_coleta[df_coleta.duplicated(subset='key', keep=False)].sort_values('key')
dup_pro = df_proprio[df_proprio.duplicated(subset='key', keep=False)].sort_values('key')

print(f'Duplicatas de chave em COLETA : {len(dup_col)} linhas')
print(f'Duplicatas de chave em PROPRIO: {len(dup_pro)} linhas')
if len(dup_col):
    display(dup_col[['key', 'documento', 'filial', 'valor_bruto']].head(20))
if len(dup_pro):
    display(dup_pro[['key', 'documento', 'filial', 'valor_bruto']].head(20))

Duplicatas de chave em COLETA : 0 linhas
Duplicatas de chave em PROPRIO: 0 linhas


## 3. Merge (outer) Coleta x Proprio

In [45]:
merged = pd.merge(
    df_coleta,
    df_proprio,
    on='key',
    how='outer',
    suffixes=('_coleta', '_proprio'),
    indicator=True,
)

print('Categorias:')
print(merged['_merge'].value_counts())
print('\nTotal de linhas no merge:', len(merged))

Categorias:
_merge
both          148
right_only     15
left_only       5
Name: count, dtype: int64

Total de linhas no merge: 168


## 4. Classificacao

- `both`        : par encontrado (Coleta + Proprio)
- `left_only`   : custo de Coleta SEM receita de Proprio correspondente
- `right_only`  : receita de Proprio SEM custo de Coleta correspondente

In [46]:
matched     = merged[merged['_merge'] == 'both'].copy()
only_coleta = merged[merged['_merge'] == 'left_only'].copy()
only_proprio = merged[merged['_merge'] == 'right_only'].copy()

print(f'Pares encontrados (both)        : {len(matched)}')
print(f'Coleta sem Proprio (left_only)  : {len(only_coleta)}')
print(f'Proprio sem Coleta (right_only) : {len(only_proprio)}')

Pares encontrados (both)        : 148
Coleta sem Proprio (left_only)  : 5
Proprio sem Coleta (right_only) : 15


In [47]:
print('=== COLETA sem PROPRIO correspondente ===')
cols = ['key', 'documento_coleta', 'filial_coleta', 'lancamento_coleta', 'valor_bruto_coleta']
only_coleta[cols].sort_values('filial_coleta')

=== COLETA sem PROPRIO correspondente ===


,key,documento_coleta,filial_coleta,lancamento_coleta,valor_bruto_coleta
159,5320,CTE-5320,G3S LONDRINA,31/03/2026,697.0
106,5267,CTE-5267,G3S MARINGA,28/03/2026,1862.0
15,13483,CTE-13483,G3S PRUDENTE,25/03/2026,714.0
26,13494,CTE-13494,G3S PRUDENTE,25/03/2026,680.0
35,13505,CTE-13505,G3S PRUDENTE,27/03/2026,2686.0


In [48]:
print('=== PROPRIO sem COLETA correspondente ===')
cols = ['key', 'documento_proprio', 'filial_proprio', 'lancamento_proprio', 'valor_bruto_proprio']
only_proprio[cols].sort_values('filial_proprio')

=== PROPRIO sem COLETA correspondente ===


,key,documento_proprio,filial_proprio,lancamento_proprio,valor_bruto_proprio
47,2499,CTF42499,G&S DOURADOS,31/03/2026,391.0
57,2509,CTF42509,G&S DOURADOS,31/03/2026,4148.0
58,2510,CTF42510,G&S DOURADOS,31/03/2026,272.0
59,2511,CTF42511,G&S DOURADOS,31/03/2026,4148.0
60,2512,CTF42512,G&S DOURADOS,31/03/2026,1479.0
61,2513,CTF42513,G&S DOURADOS,31/03/2026,4216.0
62,2514,CTF42514,G&S DOURADOS,31/03/2026,4182.0
63,2515,CTF42515,G&S DOURADOS,31/03/2026,4641.0
64,2516,CTF42516,G&S DOURADOS,31/03/2026,5610.0
74,2526,CTF42526,G&S DOURADOS,31/03/2026,204.0


## 5. Verificacao de valor (pares encontrados)

Esperamos que `valor_bruto_coleta == valor_bruto_proprio` em cada par.

In [49]:
matched['delta'] = matched['valor_bruto_proprio'] - matched['valor_bruto_coleta']
divergentes = matched[matched['delta'].abs() > 0.01]

print(f'Pares com valor IGUAL    : {(matched["delta"].abs() <= 0.01).sum()}')
print(f'Pares com valor DIVERGENTE: {len(divergentes)}')

if len(divergentes):
    cols = [
        'key',
        'documento_coleta', 'filial_coleta', 'valor_bruto_coleta',
        'documento_proprio', 'filial_proprio', 'valor_bruto_proprio',
        'delta',
    ]
    display(divergentes[cols].sort_values('delta', key=abs, ascending=False))

Pares com valor IGUAL    : 148
Pares com valor DIVERGENTE: 0


## 6. Resumo por filial

In [50]:
merged['filial_norm'] = merged['filial_coleta'].fillna(merged['filial_proprio'])
merged['filial_norm'] = (
    merged['filial_norm']
    .str.replace('G3S ', '', regex=False)
    .str.replace('G&S ', '', regex=False)
    .str.strip()
)

resumo = (
    merged.groupby('filial_norm')
    .agg(
        qtd_coleta=('valor_bruto_coleta', lambda s: s.notna().sum()),
        qtd_proprio=('valor_bruto_proprio', lambda s: s.notna().sum()),
        qtd_match=('_merge', lambda s: (s == 'both').sum()),
        qtd_so_coleta=('_merge', lambda s: (s == 'left_only').sum()),
        qtd_so_proprio=('_merge', lambda s: (s == 'right_only').sum()),
        total_coleta=('valor_bruto_coleta', 'sum'),
        total_proprio=('valor_bruto_proprio', 'sum'),
    )
    .round(2)
)
resumo['delta_total'] = (resumo['total_proprio'] - resumo['total_coleta']).round(2)
resumo.loc['TOTAL'] = resumo.sum(numeric_only=True)
resumo

,qtd_coleta,qtd_proprio,qtd_match,qtd_so_coleta,qtd_so_proprio,total_coleta,total_proprio,delta_total
filial_norm,,,,,,,,
CAMPO GRANDE,28.0,28.0,28.0,0.0,0.0,43112.0,43112.0,0.0
DOURADOS,14.0,27.0,14.0,0.0,13.0,27098.0,67473.0,40375.0
LONDRINA,14.0,13.0,13.0,1.0,0.0,9418.0,8721.0,-697.0
MARINGA,56.0,56.0,55.0,1.0,1.0,77714.2,78113.2,399.0
PRUDENTE,41.0,39.0,38.0,3.0,1.0,82610.8,81488.8,-1122.0
TOTAL,153.0,163.0,148.0,5.0,15.0,239953.0,278908.0,38955.0


## 7. Diagnostico - por que os valores nao batem

A diferenca total Proprio - Coleta deve ser explicada exatamente pela soma dos orfaos:

`delta_total = sum(valor_bruto_proprio_orfao) - sum(valor_bruto_coleta_orfao)`

Tambem destacamos:
- documentos com `lancamento` **fora do mes dominante** (provavel diferenca de competencia);
- distribuicao dos orfaos por filial.

In [51]:
total_coleta = df_coleta['valor_bruto'].sum()
total_proprio = df_proprio['valor_bruto'].sum()
delta_total = total_proprio - total_coleta

soma_so_proprio = only_proprio['valor_bruto_proprio'].sum()
soma_so_coleta = only_coleta['valor_bruto_coleta'].sum()
delta_pelos_orfaos = soma_so_proprio - soma_so_coleta

print(f'Total Frete Proprio  : R$ {total_proprio:>12,.2f}')
print(f'Total Frete Coleta   : R$ {total_coleta:>12,.2f}')
print(f'Diferenca observada  : R$ {delta_total:>12,.2f}')
print('-' * 50)
print(f'(+) Proprio sem Coleta ({len(only_proprio):>2d} doc): R$ {soma_so_proprio:>12,.2f}')
print(f'(-) Coleta sem Proprio ({len(only_coleta):>2d} doc): R$ {soma_so_coleta:>12,.2f}')
print(f'(=) Diferenca pelos orfaos     : R$ {delta_pelos_orfaos:>12,.2f}')
print('-' * 50)
bate = abs(delta_total - delta_pelos_orfaos) < 0.01
print(f'Bate? {"SIM" if bate else "NAO"} (resto = R$ {delta_total - delta_pelos_orfaos:.2f})')

Total Frete Proprio  : R$   278,908.00
Total Frete Coleta   : R$   239,953.00
Diferenca observada  : R$    38,955.00
--------------------------------------------------
(+) Proprio sem Coleta (15 doc): R$    45,594.00
(-) Coleta sem Proprio ( 5 doc): R$     6,639.00
(=) Diferenca pelos orfaos     : R$    38,955.00
--------------------------------------------------
Bate? SIM (resto = R$ 0.00)


In [52]:
print('Composicao da diferenca por filial:')
print()
contrib = (
    merged.groupby('filial_norm')
    .apply(lambda g: pd.Series({
        'so_proprio_qtd': (g['_merge'] == 'right_only').sum(),
        'so_proprio_val': g.loc[g['_merge'] == 'right_only', 'valor_bruto_proprio'].sum(),
        'so_coleta_qtd' : (g['_merge'] == 'left_only').sum(),
        'so_coleta_val' : g.loc[g['_merge'] == 'left_only', 'valor_bruto_coleta'].sum(),
    }))
    .round(2)
)
contrib['delta_filial'] = (contrib['so_proprio_val'] - contrib['so_coleta_val']).round(2)
contrib = contrib.sort_values('delta_filial', key=abs, ascending=False)
contrib.loc['TOTAL'] = contrib.sum(numeric_only=True)
contrib

Composicao da diferenca por filial:



,so_proprio_qtd,so_proprio_val,so_coleta_qtd,so_coleta_val,delta_filial
filial_norm,,,,,
DOURADOS,13.0,40375.0,0.0,0.0,40375.0
PRUDENTE,1.0,2958.0,3.0,4080.0,-1122.0
LONDRINA,0.0,0.0,1.0,697.0,-697.0
MARINGA,1.0,2261.0,1.0,1862.0,399.0
CAMPO GRANDE,0.0,0.0,0.0,0.0,0.0
TOTAL,15.0,45594.0,5.0,6639.0,38955.0


In [53]:
def parse_data(s):
    return pd.to_datetime(s, format='%d/%m/%Y', errors='coerce')

only_coleta['data_lanc'] = parse_data(only_coleta['lancamento_coleta'])
only_proprio['data_lanc'] = parse_data(only_proprio['lancamento_proprio'])

mes_dominante = parse_data(df_coleta['lancamento']).dt.to_period('M').mode().iat[0]
print(f'Mes dominante (Coleta): {mes_dominante}')
print()

fora_pro = only_proprio[only_proprio['data_lanc'].dt.to_period('M') != mes_dominante]
fora_col = only_coleta[only_coleta['data_lanc'].dt.to_period('M') != mes_dominante]

print(f'>>> Orfaos do PROPRIO fora do mes ({len(fora_pro)} doc, R$ {fora_pro["valor_bruto_proprio"].sum():,.2f}):')
display(fora_pro[['key', 'documento_proprio', 'filial_proprio', 'lancamento_proprio', 'valor_bruto_proprio']])

print(f'\n>>> Orfaos da COLETA fora do mes ({len(fora_col)} doc, R$ {fora_col["valor_bruto_coleta"].sum():,.2f}):')
display(fora_col[['key', 'documento_coleta', 'filial_coleta', 'lancamento_coleta', 'valor_bruto_coleta']])

Mes dominante (Coleta): 2026-03

>>> Orfaos do PROPRIO fora do mes (1 doc, R$ 2,958.00):


,key,documento_proprio,filial_proprio,lancamento_proprio,valor_bruto_proprio
41,13511,CTF013511,G&S PRUDENTE,08/04/2026,2958.0



>>> Orfaos da COLETA fora do mes (0 doc, R$ 0.00):


,key,documento_coleta,filial_coleta,lancamento_coleta,valor_bruto_coleta


## 7. Conclusao

In [54]:
total_coleta = df_coleta['valor_bruto'].sum()
total_proprio = df_proprio['valor_bruto'].sum()
valor_match = matched['valor_bruto_coleta'].sum()
valor_so_coleta = only_coleta['valor_bruto_coleta'].sum()
valor_so_proprio = only_proprio['valor_bruto_proprio'].sum()

cobertura_coleta = valor_match / total_coleta * 100 if total_coleta else 0
cobertura_proprio = valor_match / total_proprio * 100 if total_proprio else 0

print('=' * 60)
print('RESUMO GERAL')
print('=' * 60)
print(f'Frete Coleta   : {len(df_coleta):>4d} doc | R$ {total_coleta:>14,.2f}')
print(f'Frete Proprio  : {len(df_proprio):>4d} doc | R$ {total_proprio:>14,.2f}')
print('-' * 60)
print(f'Pares (match)  : {len(matched):>4d} doc | R$ {valor_match:>14,.2f}')
print(f'Cobertura Coleta : {cobertura_coleta:6.2f}%')
print(f'Cobertura Proprio: {cobertura_proprio:6.2f}%')
print('-' * 60)
print(f'Coleta SEM Proprio : {len(only_coleta):>3d} doc | R$ {valor_so_coleta:>14,.2f}')
print(f'Proprio SEM Coleta : {len(only_proprio):>3d} doc | R$ {valor_so_proprio:>14,.2f}')
print(f'Pares com valor divergente: {len(divergentes):>3d}')

RESUMO GERAL
Frete Coleta   :  153 doc | R$     239,953.00
Frete Proprio  :  163 doc | R$     278,908.00
------------------------------------------------------------
Pares (match)  :  148 doc | R$     233,314.00
Cobertura Coleta :  97.23%
Cobertura Proprio:  83.65%
------------------------------------------------------------
Coleta SEM Proprio :   5 doc | R$       6,639.00
Proprio SEM Coleta :  15 doc | R$      45,594.00
Pares com valor divergente:   0
